# Q5: Text Preprocessing → Tokenization → Embeddings
### Using CLIP to encode flower descriptions into vectors for text-to-image models
**Pipeline:** Raw Text → Clean → Tokenize → CLIP Encoder → 512D Embedding → Visualize

In [ ]:
# CELL 1: Install Dependencies
!pip install gradio torch torchvision transformers matplotlib numpy Pillow scikit-learn -q
print('Done!')

In [ ]:
# CELL 2: Imports
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import gradio as gr
import re
import string
from transformers import CLIPTokenizer, CLIPTextModel
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLIP_MODEL = 'openai/clip-vit-base-patch32'
MAX_TOKENS = 77
print(f'Device: {DEVICE}')
print('Imports done!')

In [ ]:
# CELL 3: Load CLIP Tokenizer and Text Encoder
print('Loading CLIP tokenizer and text encoder...')
print('This downloads ~500MB on first run')
tokenizer   = CLIPTokenizer.from_pretrained(CLIP_MODEL)
text_encoder = CLIPTextModel.from_pretrained(CLIP_MODEL).to(DEVICE)
text_encoder.eval()
print('CLIP loaded!')
print(f'Vocab size      : {tokenizer.vocab_size}')
print(f'Max token length: {tokenizer.model_max_length}')
print(f'Model device    : {DEVICE}')

In [ ]:
# CELL 4: Text Preprocessing Pipeline
def preprocess_text(text: str) -> str:
    if not text or not text.strip():
        raise ValueError('Empty text input')

    text = text.lower()
    text = re.sub(r"[^a-z0-9\s,.\'\-]", '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    if len(text) > 200:
        text = text[:200].rsplit(' ', 1)[0]
    return text

def tokenize_text(text: str) -> dict:
    tokens = tokenizer(
        text,
        padding='max_length',
        max_length=MAX_TOKENS,
        truncation=True,
        return_tensors='pt'
    )
    return tokens

def get_embedding(text: str) -> np.ndarray:
    cleaned = preprocess_text(text)
    tokens  = tokenize_text(cleaned)
    tokens  = {k: v.to(DEVICE) for k, v in tokens.items()}
    with torch.no_grad():
        output = text_encoder(**tokens)
    embedding = output.pooler_output.squeeze(0).cpu().numpy()
    return embedding


sample_text = 'A vibrant rose with rounded petals and silky texture.'
print('=== FULL PIPELINE DEMO ===')
print(f'Original text : "{sample_text}"')
cleaned = preprocess_text(sample_text)
print(f'After preprocess: "{cleaned}"')
tokens = tokenize_text(cleaned)
input_ids = tokens['input_ids'][0]
attn_mask = tokens['attention_mask'][0]
print(f'Token IDs shape : {input_ids.shape}')
print(f'Token IDs (first 15): {input_ids[:15].tolist()}')
print(f'Attention mask  : {attn_mask.tolist()[:15]}...')
print(f'Active tokens   : {attn_mask.sum().item()} / {MAX_TOKENS}')

decoded_tokens = [tokenizer.decode([t]) for t in input_ids[:attn_mask.sum().item()]]
print(f'Decoded tokens  : {decoded_tokens}')
embedding = get_embedding(sample_text)
print(f'Embedding shape : {embedding.shape}')
print(f'Embedding dtype : {embedding.dtype}')
print(f'Embedding norm  : {np.linalg.norm(embedding):.4f}')
print(f'First 8 values  : {embedding[:8].round(4)}')

In [ ]:
# CELL 5: Batch embed all flower descriptions from Q4
FLOWER_NAMES = [
    'pink primrose', 'hard-leaved pocket orchid', 'canterbury bells', 'sweet pea',
    'english marigold', 'tiger lily', 'moon orchid', 'bird of paradise', 'monkshood',
    'globe thistle', 'snapdragon', 'colt s foot', 'king protea', 'spear thistle',
    'yellow iris', 'globe-flower', 'purple coneflower', 'peruvian lily', 'balloon flower',
    'giant white arum lily', 'fire lily', 'pincushion flower', 'fritillary', 'red ginger',
    'grape hyacinth', 'corn poppy', 'prince of wales feathers', 'stemless gentian',
    'artichoke', 'sweet william', 'carnation', 'garden phlox', 'love in the mist',
    'mexican aster', 'alpine sea holly', 'ruby-lipped cattleya', 'cape flower',
    'great masterwort', 'siam tulip', 'lenten rose', 'barberton daisy', 'daffodil',
    'sword lily', 'poinsettia', 'bolero deep blue', 'wallflower', 'marigold',
    'buttercup', 'oxeye daisy', 'common dandelion', 'petunia', 'wild pansy',
    'primula', 'sunflower', 'pelargonium', 'bishop of llandaff', 'gaura', 'geranium',
    'orange dahlia', 'pink-yellow dahlia', 'cautleya spicata', 'japanese anemone',
    'black-eyed susan', 'silverbush', 'californian poppy', 'osteospermum', 'spring crocus',
    'bearded iris', 'windflower', 'tree poppy', 'gazania', 'azalea', 'water lily',
    'rose', 'thorn apple', 'morning glory', 'passion flower', 'lotus', 'toad lily',
    'anthurium', 'frangipani', 'clematis', 'hibiscus', 'columbine', 'desert-rose',
    'tree mallow', 'magnolia', 'cyclamen', 'watercress', 'canna lily', 'hippeastrum',
    'bee balm', 'ball moss', 'foxglove', 'bougainvillea', 'camellia', 'mallow',
    'mexican petunia', 'bromelia', 'blanket flower', 'trumpet creeper', 'blackberry lily'
]
import random
random.seed(42)
COLOR_WORDS   = ['vibrant', 'delicate', 'rich', 'pale', 'bright', 'deep', 'soft', 'vivid']
PETAL_WORDS   = ['rounded', 'pointed', 'layered', 'ruffled', 'slender', 'broad', 'curved']
TEXTURE_WORDS = ['silky', 'velvety', 'waxy', 'papery', 'smooth', 'textured', 'glossy']
SHAPE_WORDS   = ['trumpet-shaped', 'star-shaped', 'cup-shaped', 'bell-shaped', 'flat', 'dome-shaped']

def generate_description(flower_name):
    templates = [
        f'A {random.choice(COLOR_WORDS)} {flower_name} with {random.choice(PETAL_WORDS)} petals and {random.choice(TEXTURE_WORDS)} texture.',
        f'This {flower_name} displays {random.choice(SHAPE_WORDS)} blooms with {random.choice(TEXTURE_WORDS)} {random.choice(PETAL_WORDS)} petals.',
        f'A beautiful {flower_name} featuring {random.choice(COLOR_WORDS)} {random.choice(PETAL_WORDS)} petals in a {random.choice(SHAPE_WORDS)} form.',
    ]
    return random.choice(templates)

CLASS_DESCRIPTIONS = {i: generate_description(name) for i, name in enumerate(FLOWER_NAMES)}

print('Embedding all 102 flower descriptions...')
all_embeddings = []
for i in range(102):
    emb = get_embedding(CLASS_DESCRIPTIONS[i])
    all_embeddings.append(emb)
    if (i + 1) % 20 == 0:
        print(f'   {i+1}/102 done...')
all_embeddings = np.array(all_embeddings)
print(f'All embeddings shape: {all_embeddings.shape}')
print('Done!')

In [ ]:
# CELL 6: Visualize Embeddings with PCA
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(all_embeddings)
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum() * 100:.1f}%')

plt.figure(figsize=(14, 10))
colors = cm.tab20(np.linspace(0, 1, 102))
for i, (x, y) in enumerate(embeddings_2d):
    plt.scatter(x, y, color=colors[i], s=60, zorder=2)
    if i %  10 == 0:
        plt.annotate(FLOWER_NAMES[i], (x, y), fontsize=7, xytext=(5, 5), textcoords='offset points')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% variance)')
plt.title('CLIP Text Embeddings — Oxford 102 Flowers (PCA 2D)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


sim_matrix = cosine_similarity(all_embeddings[:20])
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(sim_matrix, cmap='coolwarm', vmin=0.8, vmax=1.0)
ax.set_xticks(range(20))
ax.set_yticks(range(20))
ax.set_xticklabels([FLOWER_NAMES[i][:12] for i in range(20)], rotation=45, ha='right', fontsize=8)
ax.set_yticklabels([FLOWER_NAMES[i][:12] for i in range(20)], fontsize=8)
plt.colorbar(im, ax=ax, label='Cosine Similarity')
plt.title('Embedding Similarity Between First 20 Flower Classes')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 7: Gradio UI — Full Text Embedding Pipeline
import gradio as gr

def run_pipeline(raw_text: str, compare_text: str):
    if not raw_text.strip():
        return '', '', '', None, 'Please enter some text!'
    try:
        cleaned = preprocess_text(raw_text)
        tokens   = tokenize_text(cleaned)
        input_ids = tokens['input_ids'][0]
        attn_mask = tokens['attention_mask'][0]
        n_active  = attn_mask.sum().item()
        decoded   = [tokenizer.decode([t]) for t in input_ids[:n_active]]
        emb1 = get_embedding(raw_text)

        sim_info = ''
        if compare_text.strip():
            emb2     = get_embedding(compare_text)
            sim      = cosine_similarity([emb1], [emb2])[0][0]
            sim_info = f'Cosine Similarity with compare text: {sim:.4f}\n({"Very similar" if sim > 0.95 else "Somewhat similar" if sim > 0.90 else "Different"})'

        sims      = cosine_similarity([emb1], all_embeddings)[0]
        top3_idx  = sims.argsort()[-3:][::-1]
        top3_info = '\n'.join([f'   {FLOWER_NAMES[i]} (sim={sims[i]:.4f})' for i in top3_idx])

        step1_out = f'Original:\n  "{raw_text}"\n\nCleaned:\n  "{cleaned}"\n\nChanges made:\n  - Lowercased\n  - Removed special chars\n  - Collapsed whitespace'
        step2_out = f'Token IDs (first 15):\n  {input_ids[:15].tolist()}\n\nActive tokens: {n_active}/{MAX_TOKENS}\n\nDecoded tokens:\n  {decoded}'
        step3_out = f'Embedding shape : {emb1.shape}\nEmbedding norm  : {np.linalg.norm(emb1):.4f}\nFirst 8 values  : {emb1[:8].round(4).tolist()}\n\nTop-3 similar flower classes:\n{top3_info}\n\n{sim_info}'

        fig, axes = plt.subplots(1, 2, figsize=(12, 3))
        axes[0].bar(range(64), emb1[:64], color='steelblue', alpha=0.8)
        axes[0].set_title('First 64 Embedding Dimensions')
        axes[0].set_xlabel('Dimension')
        axes[0].set_ylabel('Value')

        axes[1].hist(emb1, bins=40, color='coral', alpha=0.8, edgecolor='black')
        axes[1].set_title('Embedding Value Distribution')
        axes[1].set_xlabel('Value')
        axes[1].set_ylabel('Count')

        plt.suptitle(f'CLIP Embedding: "{raw_text[:50]}"', fontsize=10)
        plt.tight_layout()
        return step1_out, step2_out, step3_out, fig, 'Pipeline complete!'
    except Exception as e:
        return '', '', '', None, f'Error: {str(e)}'

with gr.Blocks(title='Text Embedding Pipeline', theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Q5: Text Preprocessing → Tokenization → Embedding Pipeline
    ### Using CLIP to encode text descriptions into 512D vectors
    """)
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### Input')
            text_input = gr.Textbox(
                label='Text Description',
                placeholder='e.g. A vibrant rose with silky rounded petals',
                value='A vibrant rose with silky rounded petals',
                lines=3
            )
            compare_input = gr.Textbox(
                label='Compare With (optional)',
                placeholder='e.g. A delicate lotus with wavy petals',
                lines=2
            )
            run_btn = gr.Button('Run Pipeline', variant='primary', size='lg')

            gr.Markdown('### Example Descriptions')
            examples = [
                'A vibrant rose with rounded silky petals',
                'A tall sunflower with broad yellow petals',
                'A delicate lotus floating on water',
                'A bright hibiscus with trumpet-shaped red blooms',
            ]
            for ex in examples:
                gr.Button(ex[:40]).click(lambda e=ex: e, outputs=text_input)

        with gr.Column(scale=2):
            status_out = gr.Textbox(label='Status', lines=1, interactive=False)
            with gr.Tab('Step 1: Preprocessing'):
                step1_out = gr.Textbox(label='Preprocessing Result', lines=8, interactive=False)
            with gr.Tab('Step 2: Tokenization'):
                step2_out = gr.Textbox(label='Tokenization Result', lines=8, interactive=False)
            with gr.Tab('Step 3: Embedding'):
                step3_out = gr.Textbox(label='Embedding Result', lines=10, interactive=False)
            embed_plot = gr.Plot(label='Embedding Visualization')

    run_btn.click(
        fn=run_pipeline,
        inputs=[text_input, compare_input],
        outputs=[step1_out, step2_out, step3_out, embed_plot, status_out]
    )

demo.launch(share=True, debug=True)